# Compare Performance

In [ ]:
from hglm.compare_analyses import load_update_all, get_path_result

# prints all experiments (useful to pick a folder)
path_result = get_path_result()
sorted(path_result.glob('exp_*'))

In [ ]:
# load data
df = load_update_all()

In [ ]:
df.head()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# === CONFIG ===
x_param = 'rough'                # or 'hotel_tr'
metrics = ['f1', 'sens', 'spec']

tab10 = plt.get_cmap('tab10')
color_map = {'AnalysisHGLM': tab10(0), 'AnalysisTFCE': tab10(3)}
label_map = {'AnalysisHGLM': 'HGLM', 'AnalysisTFCE': 'TFCE'}

# --- 1) Aggregate to unique (Analysis, seed, x_param) ---
df_agg = (df.groupby(['Analysis', 'seed', x_param], as_index=False)[metrics]
            .mean())

fig, axes = plt.subplots(2, 3, figsize=(14, 5.5), sharex='col')
for j, metric in enumerate(metrics):
    ax_top, ax_bot = axes[0, j], axes[1, j]

    # ---- TOP: per-seed lines + bold mean ----
    for analysis, sub in df_agg.groupby('Analysis'):
        color = color_map[analysis]

        for seed, g in sub.groupby('seed'):
            g = g.sort_values(x_param)
            ax_top.plot(g[x_param], g[metric], lw=1, alpha=0.6, color=color)

#         mean_curve = (sub.groupby(x_param, as_index=False)[metric]
#                         .mean()
#                         .sort_values(x_param))
#         ax_top.plot(mean_curve[x_param], mean_curve[metric],
#                     lw=3, color=color, label=label_map[analysis])

    if j == 0:
        ax_top.legend(frameon=False)
    ax_top.set_title(metric)
    ax_top.grid(True, alpha=0.4, linewidth=1.2)

    # ---- BOTTOM: HGLM - TFCE diffs (per seed) ----
    pivot = (df_agg.pivot_table(index=['seed', x_param],
                                columns='Analysis',
                                values=metric)
                   .reset_index()
                   .dropna(subset=['AnalysisHGLM', 'AnalysisTFCE'])
                   .sort_values(['seed', x_param]))
    pivot['diff'] = pivot['AnalysisHGLM'] - pivot['AnalysisTFCE']

    for seed, g in pivot.groupby('seed'):
        ax_bot.plot(g[x_param], g['diff'], lw=1, alpha=0.6, color='black')

#     diff_mean = (pivot.groupby(x_param, as_index=False)['diff']
#                        .mean()
#                        .sort_values(x_param))
#     ax_bot.plot(diff_mean[x_param], diff_mean['diff'], lw=3, color='black')

    ax_bot.axhline(0, lw=1, color='black', alpha=0.3)
    ax_bot.set_xlabel(x_param)
    ax_bot.grid(True, alpha=0.4, linewidth=1.2)

# --- Log scale only if valid ---
xmin = df_agg[x_param].min()
if xmin > 0:
    for ax_row in axes:
        for ax in ax_row:
            ax.set_xscale('log')
# else: keep linear, or use symlog:
# for ax_row in axes:
#     for ax in ax_row:
#         ax.set_xscale('symlog', linthresh=1e-3)

axes[0, 0].set_ylabel('score')
axes[1, 0].set_ylabel('HGLM - TFCE')

plt.tight_layout()
plt.show()


# Compare Computation Time

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.swarmplot(data=df[df['Analysis'] != 'AnalysisHGLM-maxF1'], x='time_sec', hue='Analysis', size=3)
plt.suptitle('Time per experiment')
plt.xlabel('Time (seconds)')
plt.gcf().set_size_inches(10, 5)